# Importing Libraries
Basic pyhton libraries are imported here

In [42]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler, LabelEncoder
import matplotlib.pyplot as plt
from scipy.stats import zscore
import plotly.express as px

Loading Dataset

In [30]:
df = pd.read_csv(r"D:\HR_comma_sep (1).csv")
df.head()

,satisfaction_level,last_evaluation,number_project,average_montly_hours,time_spend_company,Work_accident,left,promotion_last_5years,Department,salary
0,0.38,0.53,2,157,3,0,1,0,sales,low
1,0.80,0.86,5,262,6,0,1,0,sales,medium
2,0.11,0.88,7,272,4,0,1,0,sales,medium
3,0.72,0.87,5,223,5,0,1,0,sales,low
4,0.37,0.52,2,159,3,0,1,0,sales,low


Checks for the null values in the dataset

In [31]:
df.isnull().sum()

satisfaction_level       0
last_evaluation          0
number_project           0
average_montly_hours     0
time_spend_company       0
Work_accident            0
left                     0
promotion_last_5years    0
Department               0
salary                   0
dtype: int64

Extra columns are dropped here which are not effecting the target feature

In [32]:
df.drop(columns=['last_evaluation', 'Work_accident', 'left', 'promotion_last_5years'], inplace=True)
df.head()

,satisfaction_level,number_project,average_montly_hours,time_spend_company,Department,salary
0,0.38,2,157,3,sales,low
1,0.80,5,262,6,sales,medium
2,0.11,7,272,4,sales,medium
3,0.72,5,223,5,sales,low
4,0.37,2,159,3,sales,low


Handling ouliers

In [33]:
columns = ['satisfaction_level', 'number_project',
                'average_montly_hours']

for columns in columns:
    Q1 = df[columns].quantile(0.25)
    Q3 = df[columns].quantile(0.75)
    IQR = Q3 - Q1
    lower_fence = Q1 - 1.5 * IQR
    upper_fence = Q3 + 1.5 * IQR

    outliers = df[(df[columns] < lower_fence) | (df[columns] > upper_fence)]
    print(f'Outliers in {columns}: {len(outliers)}')

    df[columns] = df[columns].clip(lower=lower_fence, upper=upper_fence)

    outliers = df[(df[columns] < lower_fence) | (df[columns] > upper_fence)]
    print(f'Outliers after clipping in {columns}: {len(outliers)}')

   #  'time_spend_company'
Q1 = df['time_spend_company'].quantile(0.25)
Q3 = df['time_spend_company'].quantile(0.75)
IQR = Q3 - Q1
lower_fence = Q1 - 1.5 * IQR
upper_fence = Q3 + 1.5 * IQR
outliers = df[(df['time_spend_company'] < lower_fence) | (df['time_spend_company'] > upper_fence)]
print(f'Outliers in {columns}: {len(outliers)}')


Outliers in satisfaction_level: 0
Outliers after clipping in satisfaction_level: 0
Outliers in number_project: 0
Outliers after clipping in number_project: 0
Outliers in average_montly_hours: 0
Outliers after clipping in average_montly_hours: 0
Outliers in average_montly_hours: 1282


Department and Salary are encoded here

In [34]:

map = {'low': 20000, 'medium': 50000, 'high': 100000}
df['salary'] = df['salary'].map(map)
print(df['salary'].value_counts())

encoded_department = LabelEncoder()
df['Department'] = encoded_department.fit_transform(df['Department'])
df

salary
20000     7316
50000     6446
100000    1237
Name: count, dtype: int64


,satisfaction_level,number_project,average_montly_hours,time_spend_company,Department,salary
0,0.38,2,157,3,7,20000
1,0.80,5,262,6,7,50000
2,0.11,7,272,4,7,50000
3,0.72,5,223,5,7,20000
4,0.37,2,159,3,7,20000
...,...,...,...,...,...,...
14994,0.40,2,151,3,8,20000
14995,0.37,2,160,3,8,20000
14996,0.37,2,143,3,8,20000
14997,0.11,6,280,4,8,20000


Feature Scaling

In [39]:
scaler = MinMaxScaler()
df_normalize = pd.DataFrame(scaler.fit_transform(df), columns=df.columns)
print(f"\nNormalized Data:\n{df_normalize}")


Normalized Data:
       satisfaction_level  number_project  average_montly_hours  \
0                0.318681             0.0              0.285047   
1                0.780220             0.6              0.775701   
2                0.021978             1.0              0.822430   
3                0.692308             0.6              0.593458   
4                0.307692             0.0              0.294393   
...                   ...             ...                   ...   
14994            0.340659             0.0              0.257009   
14995            0.307692             0.0              0.299065   
14996            0.307692             0.0              0.219626   
14997            0.021978             0.8              0.859813   
14998            0.307692             0.0              0.289720   

       time_spend_company  Department  salary  
0                   0.125    0.777778   0.000  
1                   0.500    0.777778   0.375  
2                   0.250    0.77

## Visualization

Bar Chart of satisfaction by Department

In [70]:
df_bar = df.groupby('Department')['satisfaction_level'].mean().reset_index()

fig = px.bar(
    df_bar, x='Department', y='satisfaction_level',
    title='Average Satisfaction Level by Department',
    color='satisfaction_level',

    labels={'satisfaction_level': 'Satisfaction Level Reference'}
)
fig.update_traces(hovertemplate='<b>Department</b>: %{x}<br><b>Avg Satisfaction</b>: %{y:.2f}')
fig.show()

Interactive Line Chart

In [64]:
fig = px.scatter(
    df, x='average_montly_hours', y='satisfaction_level',
    color='salary',
    category_orders={'salary': ['low', 'medium', 'high']},
    hover_data=['Department', 'number_project', 'time_spend_company'],
    title='Scatter Chart for Monthly Hours with Satisfaction Level',
    labels={
        'average_montly_hours': 'Avg Monthly Hours',
        'satisfaction_level': 'Satisfaction Level',
        'salary': 'Salary Level'
    },

)
fig.show()

Bar Chart for Salary with satisfaction 

In [69]:
fig = px.bar(
    df.groupby('salary')['satisfaction_level'].mean().reset_index(),
    x='salary', y='satisfaction_level',
    category_orders={'salary': ['low', 'medium', 'high']},
    color='salary',
    title='Bar Chart for salary with Satisfaction Level',
    labels={'satisfaction_level': 'Satisfaction Level', 'salary': 'Salary Level'},
)
fig.update_traces(hovertemplate='<b>Salary</b>: %{x}<br><b>Satisfaction</b>: %{y:.2f}')
fig.show()